# Text Embeddings (Lesson 34)

Follow-along notebook for *Text embeddings* in Building with the Claude API.

**What you'll do:**
1. Set up VoyageAI and generate an embedding (the lesson's code)
2. Look at what an embedding actually is
3. Embed many chunks in one call, and see why `input_type` matters
4. A preview of the next lesson: comparing embeddings to find the chunk that answers a question

**Before you start:**
- Create a VoyageAI account and API key (free to start), then add `VOYAGE_API_KEY="..."` to your `.env` file next to `ANTHROPIC_API_KEY`.
- Voyage is a separate third-party service from Anthropic. Send it only the course's sample data, not TSG or client content, unless that use has been approved.

**Run order:** Run All after a kernel restart.

## 1. Install
`%pip` installs into the kernel's environment (your `.venv`). Run once, then you can comment it out.

In [1]:
%pip install voyageai python-dotenv

  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
   ---------------------------------------- 0.0/588.1 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/588.1 kB ? eta -:--:--
   ----------------------------------- ---- 524.3/588.1 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 588.1/588.1 kB 1.4 MB/s  0:00:00
   ---------------------------------------- 0.0/940.8 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/940.8 kB ? eta -:--:--
   --------------------------------- ------ 786.4/940.8 kB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 940.8/940.8 kB 2.1 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.6 MB 2.0 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/12.6 MB 2.1 MB/s eta 0:00:06
   ------ --------------------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Setup

The lesson names the Voyage client `client`. Here it's `voyage_client`, because later RAG lessons use Voyage and Anthropic in the same notebook and both would want the name `client`.

`max_retries` helps with Voyage's rate limits, which are very low on new accounts until a payment method is added.

In [7]:
import os
from math import sqrt
from pathlib import Path

import voyageai
from dotenv import load_dotenv
from voyageai.client import Client

load_dotenv()
print("VOYAGE_API_KEY found:", bool(os.getenv("VOYAGE_API_KEY")))

voyage_client = Client(
    api_key=os.getenv("VOYAGE_API_KEY"),
    max_retries=3,
)

# The lesson uses voyage-3-large. Voyage now lists it as a previous generation;
# voyage-4-large or voyage-4 are the newer equivalents if you want to switch.
EMBED_MODEL = "voyage-3-large"

VOYAGE_API_KEY found: True


## 3. Generate an embedding (the lesson's function)

In [8]:
load_dotenv(override=True)
api_key = os.getenv("VOYAGE_API_KEY")
if not api_key:
    raise RuntimeError("Add VOYAGE_API_KEY to your .env file and rerun this cell.")

voyage_client = Client(api_key=api_key, max_retries=3)


def generate_embedding(text, model=EMBED_MODEL, input_type="query"):
    result = voyage_client.embed([text], model=model, input_type=input_type)
    return result.embeddings[0]


embedding = generate_embedding("The team fixed 1,247 bugs this year.")
print(type(embedding), len(embedding))
print(embedding[:8])

<class 'list'> 1024
[-0.014541445299983025, -0.012054759077727795, -0.012577234767377377, -0.015379218384623528, 0.010568819008767605, 0.02965916134417057, -0.02929060347378254, 0.01017895806580782]


## 4. What's in the numbers

The lesson says each value ranges from -1 to +1. In practice they're much smaller than that: Voyage returns vectors **normalized to length 1**, so with 1,024 dimensions each value is tiny. That normalization matters in the next lesson, because it makes cosine similarity the same as a simple dot product.

No single dimension means anything you can name; the meaning is in the overall direction of the vector.

In [9]:
length = sqrt(sum(x * x for x in embedding))
print(f"dimensions : {len(embedding)}")
print(f"min / max  : {min(embedding):.4f} / {max(embedding):.4f}")
print(f"vector length (L2 norm): {length:.4f}")

dimensions : 1024
min / max  : -0.0963 / 0.1050
vector length (L2 norm): 1.0000


## 5. Embedding many chunks at once

Calling `generate_embedding` once per chunk is slow and burns through rate limits. `embed()` takes a list, so a whole document goes in one call.

**`input_type` matters:** use `"document"` for the chunks you store and `"query"` for the user's question. Voyage adds a slightly different internal prompt for each, which improves retrieval.

In [10]:
def generate_embeddings(texts, model=EMBED_MODEL, input_type="document"):
    result = voyage_client.embed(texts, model=model, input_type=input_type)
    print(f"Embedded {len(texts)} texts using {result.total_tokens} tokens")
    return result.embeddings

Load the sample report from the chunking notebook (or recreate it if `report.md` isn't in this folder), then chunk it by section:

In [11]:
import re

if Path("report.md").exists():
    report = Path("report.md").read_text(encoding="utf-8")
else:
    report = """# Northwind Labs Annual Report 2026

## Executive Summary
Northwind Labs had a strong year across all divisions. Revenue grew 18 percent, driven by new contracts in diagnostics and cloud software. Headcount rose to 412 employees.

## Medical Research Division
The medical team completed Phase II trials for NL-204, a treatment for chronic inflammation. Enrollment was briefly paused in March after a stomach bug spread through one trial site. The bug was traced to a catering supplier and resolved within two weeks.

## Software Engineering Division
Engineers shipped four major releases of the Northwind analytics platform this year. The team fixed 1,247 bugs, a 22 percent increase over last year, and cut average resolution time from nine days to four.

## Financial Overview
Total revenue reached 96 million dollars. Operating margin improved to 14 percent. Research spending was 21 million dollars.

## Outlook
Next year the company plans to expand the analytics platform into healthcare, combining both divisions.
"""
    Path("report.md").write_text(report, encoding="utf-8")


def chunk_by_section(document_text):
    return [c.strip() for c in re.split(r"\n(?=## )", document_text) if c.strip()]


chunks = chunk_by_section(report)
for i, c in enumerate(chunks):
    print(i, c.splitlines()[0])

0 # Northwind Labs Annual Report 2026
1 ## Executive Summary
2 ## Medical Research Division
3 ## Software Engineering Division
4 ## Financial Overview
5 ## Outlook


In [12]:
chunk_embeddings = generate_embeddings(chunks, input_type="document")
print(len(chunk_embeddings), "embeddings of", len(chunk_embeddings[0]), "dimensions")

Embedded 6 texts using 283 tokens
6 embeddings of 1024 dimensions


## 6. Preview: finding the right chunk

The next lesson covers this properly, but you have everything needed to try it. **Cosine similarity** measures how closely two vectors point the same way: near 1 means similar meaning, near 0 means unrelated.

In [13]:
def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (sqrt(sum(x * x for x in a)) * sqrt(sum(y * y for y in b)))


def search(question, chunks, chunk_embeddings, top_k=3):
    q_emb = generate_embedding(question, input_type="query")
    scored = sorted(
        ((cosine_similarity(q_emb, e), c) for e, c in zip(chunk_embeddings, chunks)),
        reverse=True,
    )
    for score, chunk in scored[:top_k]:
        print(f"{score:.3f}  {chunk.splitlines()[0]}")
    return scored


results = search("How many bugs did engineers fix this year?", chunks, chunk_embeddings)

0.635  ## Software Engineering Division
0.441  ## Executive Summary
0.388  ## Medical Research Division


The Software Engineering section should rank first, ahead of the Medical section, even though both contain "bug". Embeddings capture *engineers* + *fixed* + *bugs* together as a software meaning.

Now try a question that shares **no keywords** with the answer. Keyword search would struggle here; semantic search shouldn't:

In [14]:
results = search("How profitable was the company?", chunks, chunk_embeddings)

0.559  ## Financial Overview
0.456  ## Executive Summary
0.408  ## Medical Research Division


## 7. Meaning vs. wording
Compare sentence pairs directly. One batch call embeds all of them.

In [17]:
pairs = [
    ("Engineers fixed many bugs in the software.", "The team resolved over a thousand code defects."),
    ("Engineers fixed many bugs in the software.", "A stomach bug spread through the trial site."),
    ("Engineers fixed many bugs in the software.", "Operating margin improved to 14 percent."),
]

texts = [t for pair in pairs for t in pair]
embs = generate_embeddings(texts, input_type="document")

for i, (a, b) in enumerate(pairs):
    score = cosine_similarity(embs[2 * i], embs[2 * i + 1])
    print(f"{score:.3f}  {a!r}\n       vs {b!r}\n")

Embedded 6 texts using 45 tokens
0.863  'Engineers fixed many bugs in the software.'
       vs 'The team resolved over a thousand code defects.'

0.677  'Engineers fixed many bugs in the software.'
       vs 'A stomach bug spread through the trial site.'

0.666  'Engineers fixed many bugs in the software.'
       vs 'Operating margin improved to 14 percent.'



The first pair should score highest despite sharing almost no words, and the "stomach bug" pair should score lower despite sharing the word *bug*. That's the case for semantic search over keyword matching.

## Takeaways
- An embedding is a list of floats representing the meaning of a text; individual dimensions aren't interpretable.
- Anthropic doesn't offer embeddings; Voyage is the recommended provider and needs its own API key.
- Embed chunks in batches with `input_type="document"`, and questions with `input_type="query"`.
- Similar meaning → similar direction → high cosine similarity. That's the core of the next lesson.